# Compare our output to the Thrasher implementation

This notebook compares downscaled data when running our approach trained on (1) ERA5 and (2) PGMF. We then take (2) and are able to compare it to the NEX-GDDP dataset which was trained on the same dataset. These two should be ~similar since we are largely just porting over the implementation into Python from NCL, and have implemented it largely the same.

First we do a baseline comparison between our implementation using the two different training datasets

In [ ]:
import math

import boto3
import icechunk
import matplotlib.pyplot as plt
import xarray as xr

from srm import catalog

In [ ]:
def open_icechunk(path):
    bucket, prefix = path.replace("s3://", "").split("/", 1)
    storage = icechunk.s3_storage(bucket=bucket, prefix=prefix)
    repo = icechunk.Repository.open(storage)
    session = repo.readonly_session("main")

    ds = xr.open_dataset(session.store, engine="zarr", chunks={})
    return ds

In [ ]:
def get_fname(var, scenario, version="test015_benchmark", ens="007"):
    fname = (
        "s3://carbonplan-scratch/srm/outputs/qa/"
        + version
        + "/"
        + scenario
        + "/CESM2-WACCM/"
        + var
        + "/"
        + ens
        + "/lat-35.0to-22.0_lon16.0to33.0/gdex-gmf/*/*/"
        + scenario
        + ".icechunk/"
    )

    return fname

In [ ]:
def resolve_s3_glob(path):
    """Resolve a single * wildcard in an S3 path to a real path."""
    bucket, prefix = path.replace("s3://", "").split("/", 1)

    before, after = prefix.split("*/", 1)

    s3 = boto3.client("s3")
    response = s3.list_objects_v2(Bucket=bucket, Prefix=before, Delimiter="/")

    matches = [f"s3://{bucket}/{cp['Prefix']}{after}" for cp in response.get("CommonPrefixes", [])]

    if not matches:
        raise ValueError(f"No S3 paths matched: {path}")
    if len(matches) > 1:
        raise ValueError(f"Multiple matches: {matches}")

    return matches[0]

In [ ]:
# read in the dataset we made training to ERA5
# read in the dataset we made training to PGMF
# compare the two datasets in the mean tasmax and the 99p tasmax

Then we do the more apples-to-apples comparison between nex-gddp and our PGMF implementation.

In [2]:
# read in the nex-gddp dataset (Thrasher et al (2022)'s implementation, trained on PGMF)
# compare the nex-gddp and our PGMF implementation (mean tasmax and the 99p tasmax)

In [ ]:
We should also look at individual timeseries of a year of actual data to confirm 

Then we repeat this for other variables and thresholds.